# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/k9Sx3CC/flyrank-internship-test/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This project is a binary classification task. The objective is to classify each content page as either likely to decline or not decline based on observable SEO signals. The prediction helps prioritize which pages should be reviewed for content refresh. Although the output can later be used to rank pages, the underlying learning problem is binary classification.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create the binary target
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

print("ML Task: Binary Classification")
print("Classes:")
print(df["is_declining"].value_counts().rename(index={0:"Not Declining",1:"Declining"}))

print("\nClass distribution:")
print((df["is_declining"].value_counts(normalize=True)*100).round(2).astype(str) + "%")

ML Task: Binary Classification
Classes:
is_declining
Declining        16262
Not Declining    13738
Name: count, dtype: int64

Class distribution:
is_declining
1    54.21%
0    45.79%
Name: proportion, dtype: object


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target is whether a content page is declining.

In the starter dataset, the label is derived from the observed value of `trend_direction`. Pages labeled **"down"** are assigned a value of 1 (declining), while all other pages are assigned 0. This is a defined rule based on observed search performance rather than manually annotated labels.

The model is trained only on observable features that would have been available before the outcome, avoiding information leakage.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Define target
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

print("Target variable: is_declining")
print()

print(df[["trend_direction", "is_declining"]].head(10))

print("\nLabel counts:")
print(df.groupby("trend_direction")["is_declining"].count())

print("\nOverall declining rate:",
      round(df["is_declining"].mean(), 3))

Target variable: is_declining

  trend_direction  is_declining
0            down             1
1            down             1
2            down             1
3          stable             0
4            down             1
5            down             1
6            down             1
7          stable             0
8            down             1
9            down             1

Label counts:
trend_direction
down      16262
flat       1152
new        2236
stable     5962
up         4388
Name: is_declining, dtype: int64

Overall declining rate: 0.542


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The primary evaluation metric is Precision@50.

This metric measures the proportion of truly declining pages among the top 50 pages recommended for review. Since SEO teams typically have limited time and resources, accurately prioritizing the highest-risk pages is more valuable than maximizing overall accuracy.

A Precision@50 substantially higher than the baseline rule indicates better decision support.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import json
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)
# Load model results
with open("outputs/model_results.json", "r") as f:
    results = json.load(f)

baseline = results["baseline"]["baseline_precision_at_50"]
rf = results["models"]["random_forest"]["precision_at_50"]

print(f"Baseline Precision@50: {baseline:.2f}")
print(f"Random Forest Precision@50: {rf:.2f}")
print(f"Improvement: {rf - baseline:.2f}")
print(f"Relative Improvement: {rf / baseline:.2f}x")
print("Declining pages:", df["is_declining"].sum())
print("Non-declining pages:", len(df) - df["is_declining"].sum())
print("Declining rate:", round(df["is_declining"].mean(), 3))

Baseline Precision@50: 0.24
Random Forest Precision@50: 0.74
Improvement: 0.50
Relative Improvement: 3.08x
Declining pages: 16262
Non-declining pages: 13738
Declining rate: 0.542


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Each row in the dataset represents one content page.

Every row contains observable SEO features such as impressions, CTR, average ranking position, content age, search volume, and engagement metrics. The machine learning model predicts whether that individual page is likely to experience declining search performance.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Binary target
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

print("Dataset shape:", df.shape)
print("One row represents one content page.\n")

cols = [
    "content_id",
    "client_id",
    "impressions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "trend_direction",
    "is_declining"
]

print(df[cols].head())

print("\nDeclining rate:", round(df["is_declining"].mean(), 3))

Dataset shape: (30000, 45)
One row represents one content page.

             content_id          client_id  impressions_90d  avg_position  \
0  content_304f48230142  client_f369cb89fc             3803          10.6   
1  content_a1fb4e703a9e  client_4e07408562            15320          20.3   
2  content_9aa793d4d895  client_7f2253d7e2            12581          36.5   
3  content_331d6c4de07b  client_19581e27de            11751           6.2   
4  content_d99b7a2d90ca  client_3fdba35f04            19140          44.0   

    ctr  content_age_days  days_since_last_update trend_direction  \
0  0.76               187                      20            down   
1  0.05               445                      25            down   
2  0.09               141                      20            down   
3  0.49               463                      22          stable   
4  0.13               263                      14            down   

   is_declining  
0             1  
1             1  
2  

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A simple rule such as "refresh pages older than 180 days with many impressions" captures only one or two conditions.

Content performance depends on many interacting factors, including impressions, ranking position, CTR, engagement, content age, and query diversity. These relationships are often nonlinear and difficult to express with a few if-statements.

Machine learning can learn these interactions directly from historical observations while still being evaluated honestly using client-level validation.

Also, A fixed rule cannot capture the complex interactions between multiple features such as impressions, clicks, search volume, word count, and historical trends. Machine learning learns these patterns from data and ranks pages more accurately than a simple rule.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.